# Notebook for preprocessing
Run notebooks in the following order:\
`preprocessing.ipynb -> imputation.R -> dimensionality_reduction.R -> gmm_clustering.ipynb`

In [1]:
# Update path and working directory with project root
import ast
import itertools
import typing
import os
import sys
from pathlib import Path

project_path = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file())
os.chdir(project_path)
sys.path.insert(0, str(project_path))

In [2]:
# Imports
import itertools
import typing
from pathlib import Path
from typing import cast

import pandas as pd
from pandas.io.stata import StataReader

from utils.preprocessing_utils import (
    convert_data_types,
    create_meta,
    create_missing_info,
    get_description,
    impute_allowed_skip,
    remove_variables,
    sort_groups_meta,
    sort_types_meta,
)
from utils.rules_utils import EquivalenceRule, ImplicationRule, enforce_rules
from utils.saving_utils import DatasetBundle

pd.options.mode.copy_on_write = True
pd.options.future.infer_string = True

# Set outputs folder
output_folder = Path("data/preprocessed")
output_folder.mkdir(parents=True, exist_ok=True)

In [3]:
# Read in data
data_reader = pd.read_stata(
    Path("data/raw/ESA1821_missing.dta"), iterator=True, convert_categoricals=False, convert_missing=False
)
data_reader = typing.cast(StataReader, data_reader)

value_labels = data_reader.value_labels()
var_labels = data_reader.variable_labels()
with data_reader:
    data_full = DatasetBundle(data_reader.read())
    data_full.meta = cast(None, data_full.meta)
    data_full.na = cast(None, data_full.na)

# Fix mis-coded missing values: `amp_12` and `Miss_Abh_am` were not collected/derived in
# 2021 but are coded as allowed skips (-1) instead of not included (-5) there. Without this
# correction the whole 2021 wave would be imputed as "no amphetamine use", which in turn
# contradicts the observed `fr12_au1` and drops every 2021 amphetamine user.
# Must run before `create_missing_info`; `zeit` is still numeric at this point.
is_2021 = data_full.data["zeit"].eq(2021)
assert data_full.data.loc[is_2021, ["amp_12", "Miss_Abh_am"]].eq(-1).all(axis=None)
data_full.data.loc[is_2021, ["amp_12", "Miss_Abh_am"]] = -5

# Create meta-information about variables and missingness information
data_full = create_meta(data_full, var_labels, value_labels)
data_full = create_missing_info(data_full, na_labels={-1: "allowed skip", -5: "not included", -9: "no answer"})
data_full.meta = cast(pd.DataFrame, data_full.meta)
data_full.na = cast(pd.DataFrame, data_full.na)

# An allowed skip can never cover an entire wave - that would be a not included in disguise
for year, idx in data_full.data.groupby("zeit").groups.items():
    all_skip = data_full.na.loc[idx].eq("allowed skip").all()
    assert not all_skip.any(), (year, all_skip[all_skip].index.tolist())

In [4]:
# Impute allowed skips with negative values
imp_val = dict.fromkeys(
    [
        "audit", "Anzahl", "fag_sch", "dsm_ta", "psecdi_sch", "hnb_fag_sch", "hasch_12", "sds_c", "dsm_ha", "dsm_ha_m",
        "Miss_Abh_ha", "ecsta_12", "lsd_12", "heroi_12", "andop_12", "kokai_12", "sds_k", "dsm_kc", "dsm_kc_m",
        "Miss_Abh_kc", "crack_12", "pilz_12", "nps_12", "amp_12", "sds_a", "dsm_au", "dsm_au_m", "Miss_Abh_am",
        "meth_12", "rist", "rist_noa", "pomi", "anr_12", "anr_30", "app_12", "app_30", "anab_12", "anab_30", "schm_30",
        "schl_30", "dsm_schl", "dsm_schl_m", "ber_30", "dsm_ber", "dsm_ber_m", "depr_30", "neur_30",
    ],
    0,
)  | dict.fromkeys(
    [
        "fr12_ha1", "fr12_ec1", "fr12_ls1", "fr12_he1", "fr12_ao1", "fr12_ko1", "fr12_cr1", "fr12_pi1", "fr12_nps1",
        "fr12_au1", "fr12_meth1",
    ],
    1,
)  # fmt: skip
data_full = impute_allowed_skip(data_full, imp_val=imp_val)
data_full.meta = cast(pd.DataFrame, data_full.meta)
data_full.na = cast(pd.DataFrame, data_full.na)

Imputed 57162 allowed skips.


In [6]:
# Impute using structural rules and drop invalid combinations

# implied by one month abstinence from alcohol
implied_alcohol = {
    "vx210": [0, 1, 2], "wein30gr": 0, "mish30gr": 0, "bier30gr": 0, "bierkons": 0, "weinkons": 0, "spirkons": 0,
    "mishkons": 0, "binge": 1, "alk30gr": 0, "alk30kat": 0,
}  # fmt: skip

# In the following `implies` is always abstinence from a substance `is_implied` are conclusions that necessarily follow.
rules_alcohol = [
    # one year
    ImplicationRule(implies={"vx210": [0, 1]}, is_implied={"audit": 0, "hazar": 0, "harmf": 0}),
    # one month
    ImplicationRule(implies={"vx210": [0, 1, 2]}, is_implied=implied_alcohol),
    ImplicationRule(implies={"alk30gr": 0}, is_implied=implied_alcohol),
    ImplicationRule(implies={"alk30kat": 0}, is_implied=implied_alcohol),
    EquivalenceRule({"bier30gr": 0, "bierkons": 0}),
    EquivalenceRule({"wein30gr": 0, "weinkons": 0}),
    EquivalenceRule({"mish30gr": 0, "mishkons": 0}),
]

rules_nicotine = [
    ImplicationRule(implies={"typ2": [1, 2]}, is_implied={"Anzahl": 0, "fag_sch": 0, "dsm_ta": 0}),
    EquivalenceRule({"hnb30": 0, "typ_hnb": 0}),
    EquivalenceRule({"hnb30": 1, "typ_hnb": 1}),
]

rules_drugs = [
    # one year
    EquivalenceRule({"hasch_12": 0, "fr12_ha1": 1}),
    EquivalenceRule({"ecsta_12": 0, "fr12_ec1": 1}),
    EquivalenceRule({"lsd_12": 0, "fr12_ls1": 1}),
    EquivalenceRule({"heroi_12": 0, "fr12_he1": 1}),
    EquivalenceRule({"andop_12": 0, "fr12_ao1": 1}),
    EquivalenceRule({"kokai_12": 0, "fr12_ko1": 1}),
    EquivalenceRule({"crack_12": 0, "fr12_cr1": 1}),
    EquivalenceRule({"pilz_12": 0, "fr12_pi1": 1}),
    EquivalenceRule({"nps_12": 0, "fr12_nps1": 1}),
    EquivalenceRule({"amp_12": 0, "fr12_au1": 1}),
    EquivalenceRule({"meth_12": 0, "fr12_meth1": 1}),

    # No rules for Miss_Abh_ha/kc/am: the variable is exactly max(dsm_*, dsm_*_m), so the rules were
    # provably no-ops (0 imputed, 0 dropped) and the variables are dropped as collinear in cell 12.
]  # fmt: skip

rules_medication = [
    # one year
    ImplicationRule(implies={"anr_12": 0}, is_implied={"anr_30": 0, "m_anr": 0, "t_anr": 0}),
    ImplicationRule(implies={"app_12": 0}, is_implied={"app_30": 0, "m_app": 0, "t_app": 0}),
    ImplicationRule(implies={"anab_12": 0}, is_implied={"anab_30": 0, "m_anab": 0, "t_anab": 0}),

    # one month
    ImplicationRule(implies={"anr_30": 0}, is_implied={"m_anr": 0, "t_anr": 0}),
    ImplicationRule(implies={"app_30": 0}, is_implied={"m_app": 0, "t_app": 0}),
    ImplicationRule(implies={"anab_30": 0}, is_implied={"m_anab": 0, "t_anab": 0}),
    ImplicationRule(implies={"schm_30": 0}, is_implied={"m_schm": 0, "t_schm": 0}),
    ImplicationRule(implies={"schl_30": 0}, is_implied={"m_schl": 0, "t_schl": 0}),
    ImplicationRule(implies={"ber_30": 0}, is_implied={"m_ber": 0, "t_ber": 0}),
    ImplicationRule(implies={"depr_30": 0}, is_implied={"m_depr": 0, "t_depr": 0}),
    ImplicationRule(implies={"neur_30": 0}, is_implied={"m_neur": 0, "t_neur": 0}),

    # one week
    ImplicationRule(implies={"m_anr": 0}, is_implied={"t_anr": 0}),
    ImplicationRule(implies={"m_app": 0}, is_implied={"t_app": 0}),
    ImplicationRule(implies={"m_anab": 0}, is_implied={"t_anab": 0}),
    ImplicationRule(implies={"m_schm": 0}, is_implied={"t_schm": 0}),
    ImplicationRule(implies={"m_schl": 0}, is_implied={"t_schl": 0}),
    ImplicationRule(implies={"m_ber": 0}, is_implied={"t_ber": 0}),
    ImplicationRule(implies={"m_depr": 0}, is_implied={"t_depr": 0}),
    ImplicationRule(implies={"m_neur": 0}, is_implied={"t_neur": 0}),
]  # fmt: skip

rules = [*rules_alcohol, *rules_nicotine, *rules_drugs, *rules_medication]
data_full = enforce_rules(data_full, rules=rules, na_strategy="impute", failure_strategy="drop", verbose=True)
data_full.meta = cast(pd.DataFrame, data_full.meta)
data_full.na = cast(pd.DataFrame, data_full.na)

2937 imputed, 4 dropped, used rule ImplicationRule(implies={'vx210': [0, 1]}, is_implied={'audit': 0, 'hazar': 0,...
40 imputed, 39 dropped, used rule ImplicationRule(implies={'vx210': [0, 1, 2]}, is_implied={'vx210': [0, 1, 2],...
0 imputed, 0 dropped, used rule ImplicationRule(implies={'alk30gr': 0}, is_implied={'vx210': [0, 1, 2],...
0 imputed, 0 dropped, used rule ImplicationRule(implies={'alk30kat': 0}, is_implied={'vx210': [0, 1, 2],...
0 imputed, 0 dropped, used rule EquivalenceRule(equivalent={'bier30gr': 0, 'bierkons': 0})
0 imputed, 0 dropped, used rule EquivalenceRule(equivalent={'wein30gr': 0, 'weinkons': 0})
0 imputed, 0 dropped, used rule EquivalenceRule(equivalent={'mish30gr': 0, 'mishkons': 0})
14085 imputed, 59 dropped, used rule ImplicationRule(implies={'typ2': [1, 2]}, is_implied={'Anzahl': 0, 'fag_sch': 0,...
0 imputed, 0 dropped, used rule EquivalenceRule(equivalent={'hnb30': 0, 'typ_hnb': 0})
0 imputed, 0 dropped, used rule EquivalenceRule(equivalent={'hnb30': 1, 

In [7]:
# Define variable groups
misc_vars = list(data_full.data.columns[:6])
socioecon_vars = list(data_full.data.columns[6:19])
alcohol_vars = list(data_full.data.columns[19:33])
nicotine_vars = list(data_full.data.columns[33:43])
drugs_vars = list(data_full.data.columns[43:77])
medication_vars = list(data_full.data.columns[77:])

# The groups are defined positionally, so guard against a changed column order in the source file
assert [misc_vars[0], socioecon_vars[0], alcohol_vars[0], nicotine_vars[0], drugs_vars[0], medication_vars[0]] == [
    "lfd", "ges", "vx210", "typ2", "hasch_12", "rist"
]
assert medication_vars[-1] == "t_neur"

# Assign group names
groups_by_name = {
    "misc": misc_vars, "socioecon": socioecon_vars, "alcohol": alcohol_vars, "nicotine": nicotine_vars,
    "drugs": drugs_vars, "medication": medication_vars,
}  # fmt: skip

# Define variable types
continuous_vars = [
    "wichtn", "alter", "hh", "hh14", "wein30gr", "mish30gr", "bier30gr", "audit", "alk30gr", "Anzahl", "sds_c", "sds_k",
    "sds_a", "pomi",
]  # fmt: skip

ordinal_vars = [
    "altq", "hne", "vx210", "binge", "alk30kat", "fag_sch", "psecdi_sch", "hnb_fag_sch", "fr12_ha1", "fr12_ec1",
    "fr12_ls1", "fr12_he1", "fr12_ao1", "fr12_ko1", "fr12_cr1", "fr12_pi1", "fr12_nps1", "fr12_au1", "fr12_meth1",
    "schm_kat", "schl_kat", "ber_kat", "anr_kat", "app_kat", "depr_kat", "neur_kat", "anab_kat",
]  # fmt: skip

binary_vars = [
    var
    for var in data_full.data.columns
    if var not in itertools.chain(ordinal_vars, continuous_vars) and data_full.data[var].nunique(dropna=True) == 2
    # and set(value_labels.get(var, {}).values()) in [set(["nein", "ja"]), set(["Nein", "Ja"])]
]

nominal_vars = [
    var for var in data_full.data.columns if var not in itertools.chain(ordinal_vars, continuous_vars, binary_vars)
]

# Assign type names
types_by_name = {"continuous": continuous_vars, "binary": binary_vars, "nominal": nominal_vars, "ordinal": ordinal_vars}

# Add group and type info to information about variables
data_full = sort_groups_meta(data_full, groups_by_name)
data_full = sort_types_meta(data_full, types_by_name)
data_full.meta = cast(pd.DataFrame, data_full.meta)
data_full.na = cast(pd.DataFrame, data_full.na)

In [8]:
# Convert variables to categories (instead of letting StataReader do this, hence convert_categoricals=False in
# pd.read_stata)
data_full = convert_data_types(data_full)
data_full.meta = cast(pd.DataFrame, data_full.meta)
data_full.na = cast(pd.DataFrame, data_full.na)

In [9]:
# Recode ..._30, m_..., t_... variables into one categorical variable
medication_labels = {
    "0": "0x letzte 30 Tage",
    "1": get_description(data_full.meta, "anr_30", slice=slice(1, None)),
    "2": get_description(data_full.meta, "m_anr", slice=slice(1, None)),
    "3": get_description(data_full.meta, "t_anr", slice=slice(1, None)),
}

for med in data_full.meta["name"].str.extractall("^t_(.*)$")[0]:
    # Create new categorical variable
    data_full.data = data_full.data.assign(
        **{
            f"{med}_kat": lambda x: pd.Series(
                x[f"{med}_30"]
                .apply(lambda _: pd.NA)
                .case_when(
                    [
                        (x[f"t_{med}"].eq("1"), "3"),
                        (x[f"m_{med}"].eq("1"), "2"),
                        (x[f"{med}_30"].eq("1"), "1"),
                        (x[f"{med}_30"].eq("0"), "0"),
                    ]
                ),
                dtype=pd.CategoricalDtype(categories=["0", "1", "2", "3"], ordered=True),
            )
        }
    )

    # Add meta information
    med_meta = pd.DataFrame(
        {
            "name": f"{med}_kat",
            "description": get_description(data_full.meta, f"t_{med}", slice=0),
            "labels": [repr(medication_labels)],
            "group": "medication",
            "type": "ordinal",
        }
    ).astype(data_full.meta.dtypes.to_dict())
    data_full.meta = pd.concat([data_full.meta, med_meta], ignore_index=True)

    # Add missingness information
    data_full.na = data_full.na.assign(
        **{
            f"{med}_kat": data_full.data[f"{med}_kat"]
            .map(lambda x: "no answer" if pd.isna(x) else "observed")
            .astype(pd.CategoricalDtype(categories=["observed", "allowed skip", "not included", "no answer"]))
        }
    )

In [10]:
# Recode categories

# Retain the 10 most common cities and lump the others into one category (more than 10 if there are ties).
# `na_action="ignore"` matters: without it pandas passes NaN to the mapper and silently recodes it to "other".
top_cities = data_full.data["stadt_bl"].value_counts().nlargest(n=10, keep="all").index
data_full.data = data_full.data.assign(
    **{
        "stadt_bl": lambda x: (
            x["stadt_bl"].map(lambda val: val if val in top_cities else "other", na_action="ignore").astype("category")
        ),
    }
)

# Merge rare categories of ordinal variables. Categories with a handful of observations carry extreme
# inertia in the MFA (a single respondent can define a dimension) and cannot be fitted by polr/polyreg
# in MICE.
min_category_count = 30


def collapse_rare_categories(
    series: pd.Series, wave: pd.Series, is_observed: pd.Series, min_count: int
) -> pd.Series:
    """Merge the highest categories downwards until every retained category is large enough.

    Only waves in which the variable was actually *observed* are considered. The structural rules
    fill variables in waves where they were never collected (`fag_sch` gets 7011 rule-imputed zeros
    in 2018), and counting those waves would collapse the scale down to two categories.
    """
    categories = list(series.cat.categories)
    frame = pd.DataFrame({"value": series, "wave": wave, "observed": is_observed})
    counts = [
        {cat: int(part["value"].eq(cat).sum()) for cat in categories}
        for _, part in frame.groupby("wave", observed=True)
        if part["observed"].any()
    ]
    if not counts:
        return series

    kept = list(categories)
    while len(kept) > 2 and min(count[kept[-1]] for count in counts) < min_count:
        for count in counts:
            count[kept[-2]] += count[kept[-1]]
        kept = kept[:-1]
    if len(kept) == len(categories):
        return series

    mapping = {cat: (cat if cat in kept else kept[-1]) for cat in categories}
    return series.map(mapping, na_action="ignore").astype(pd.CategoricalDtype(categories=kept, ordered=True))


collapsed = {
    var: collapse_rare_categories(
        data_full.data[var], data_full.data["zeit"], data_full.na[var].eq("observed"), min_category_count
    )
    for var in ordinal_vars
    if isinstance(data_full.data.get(var, pd.Series(dtype="float64")).dtype, pd.CategoricalDtype)
}
# Keep only the variables that actually changed - otherwise the label sync below would overwrite the
# labels of untouched variables, including the medication_labels built in cell 8.
collapsed = {
    var: series
    for var, series in collapsed.items()
    if len(series.cat.categories) < len(data_full.data[var].cat.categories)
}
data_full.data = data_full.data.assign(**collapsed)

# Keep the stored value labels in sync with the merged categories. The category kept after a merge is
# always the lower bound ("mindestens einmal im Monat"), so its label stays semantically correct.
# Read the labels back from meta, not from `value_labels`, so the medication labels that cell 8 builds
# for the `_kat` variables survive.
for var, series in collapsed.items():
    stored = data_full.meta.loc[data_full.meta["name"].eq(var), "labels"].item()
    try:
        labels = {str(key): str(value) for key, value in ast.literal_eval(stored).items()}
    except (ValueError, SyntaxError, AttributeError):
        labels = {str(code): str(label) for code, label in value_labels.get(var, {}).items()}
    data_full.meta.loc[data_full.meta["name"].eq(var), "labels"] = repr(
        {cat: labels.get(cat, cat) for cat in series.cat.categories}
    )

print("Collapsed: " + ", ".join(f"{var} -> {len(s.cat.categories)}" for var, s in collapsed.items()))

Collapsed: alk30kat -> 4, fag_sch -> 5, psecdi_sch -> 4, hnb_fag_sch -> 2, fr12_ec1 -> 2, fr12_ls1 -> 2, fr12_he1 -> 2, fr12_ao1 -> 2, fr12_ko1 -> 2, fr12_cr1 -> 2, fr12_pi1 -> 2, fr12_nps1 -> 2, fr12_au1 -> 3, fr12_meth1 -> 2, anr_kat -> 3, app_kat -> 2, anab_kat -> 2


In [11]:
# Manually impute some non-responses because of MICE bugs or collinearities

# Impute mushroom use less than once a month for three mushroom users
missing_fr12_pi1 = data_full.na["fr12_pi1"].eq("no answer")
assert missing_fr12_pi1.sum() == 3  # three missing observations
assert data_full.data.loc[missing_fr12_pi1, "fr12_pi1"].isna().all()
assert data_full.data.loc[missing_fr12_pi1, "pilz_12"].eq("1").all()  # all are mushroom users in the last 12 months
data_full.data.loc[missing_fr12_pi1, "fr12_pi1"] = "2"
data_full.na["fr12_pi1"] = data_full.na["fr12_pi1"].cat.add_categories("manually imputed")
data_full.na.loc[missing_fr12_pi1, "fr12_pi1"] = "manually imputed"

# Impute single amphetamine non-response with negative consumption
amphetamine_vars = ["dsm_au", "dsm_au_m", "Miss_Abh_am"]
missing_amphetamine = data_full.na[amphetamine_vars].eq("no answer")
missing_amphetamine_any = missing_amphetamine.any(axis=1)
assert missing_amphetamine_any.sum() == 1  # one missing observation
assert missing_amphetamine.loc[missing_amphetamine_any].all(axis=None) and data_full.data.loc[
    missing_amphetamine_any, amphetamine_vars
].isna().all(axis=None)  # all three variables are jointly missing
data_full.data.loc[missing_amphetamine_any, amphetamine_vars] = "0"
data_full.na[amphetamine_vars] = data_full.na[amphetamine_vars].apply(
    lambda x: x.cat.add_categories("manually imputed")
)
data_full.na.loc[missing_amphetamine_any, amphetamine_vars] = "manually imputed"

In [12]:
# Discovery/validation split
discov_mask = data_full.data["zeit"].eq("2018")  # Discovery data is ESA 2018 and validation data is ESA 2021
assert discov_mask.any() and not discov_mask.all()  # a mismatched label would silently produce an empty split

data_discov_full = DatasetBundle(data_full.data[discov_mask], data_full.meta, data_full.na[discov_mask])
data_discov_full.meta = cast(pd.DataFrame, data_discov_full.meta)
data_discov_full.na = cast(pd.DataFrame, data_discov_full.na)
data_discov_full.data = data_discov_full.data.reset_index(drop=True)
data_discov_full.na = data_discov_full.na.reset_index(drop=True)

data_valid_full = DatasetBundle(data_full.data[~discov_mask], data_full.meta, data_full.na[~discov_mask])
data_valid_full.meta = cast(pd.DataFrame, data_valid_full.meta)
data_valid_full.na = cast(pd.DataFrame, data_valid_full.na)
data_valid_full.data = data_valid_full.data.reset_index(drop=True)
data_valid_full.na = data_valid_full.na.reset_index(drop=True)

In [13]:
# Remove variables
excluded_vars = [
    # alcohol
    "alk30gr",

    # nicotine
    "hnb30",

    # drugs
    "hasch_12", "ecsta_12", "lsd_12", "heroi_12", "andop_12", "kokai_12", "crack_12", "pilz_12", "nps_12", "amp_12",
    "meth_12",

    # drugs: Miss_Abh_* is exactly max(dsm_*, dsm_*_m) in every observed row, i.e. perfectly collinear
    "Miss_Abh_ha", "Miss_Abh_kc", "Miss_Abh_am",

    # drugs: still degenerate after collapsing categories (2, 2 and 6 users in total) - remove this block
    # to keep heroin, crack and methamphetamine in the analysis
    "fr12_he1", "fr12_cr1", "fr12_meth1",

    # medication
    "m_anr", "t_anr", "anr_30", "m_app", "t_app", "app_30", "m_anab", "t_anab", "anab_30", "schm_30", "m_schm",
    "t_schm", "schl_30", "m_schl", "t_schl", "ber_30", "m_ber", "t_ber", "depr_30", "m_depr", "t_depr", "neur_30",
    "m_neur", "t_neur",
]  # fmt: skip

# Variables removed because they are not included/NA in one dataset
not_discov_vars = data_full.na[discov_mask].loc[:, lambda x: ~x.eq("observed").any()].columns
not_valid_vars = data_full.na[~discov_mask].loc[:, lambda x: ~x.eq("observed").any()].columns

data_discov_reduced = remove_variables(
    data_discov_full,
    drop_vars=[*[x for x in misc_vars if x != "wichtn"], *socioecon_vars, *not_discov_vars, *excluded_vars],
)
data_discov_reduced.meta = cast(pd.DataFrame, data_discov_reduced.meta)
data_discov_reduced.na = cast(pd.DataFrame, data_discov_reduced.na)

data_valid_reduced = remove_variables(
    data_valid_full,
    drop_vars=[*[x for x in misc_vars if x != "wichtn"], *socioecon_vars, *not_valid_vars, *excluded_vars],
)
data_valid_reduced.meta = cast(pd.DataFrame, data_valid_reduced.meta)
data_valid_reduced.na = cast(pd.DataFrame, data_valid_reduced.na)


# Add "included" column to the meta-information of every bundle
def included_in(x) -> str:
    in_discov = x in data_discov_reduced.data.columns.to_list()
    in_valid = x in data_valid_reduced.data.columns.to_list()
    if in_discov and in_valid:
        return "both"
    elif in_discov:
        return "discovery"
    elif in_valid:
        return "validation"
    else:
        return "neither"


for bundle in (data_full, data_discov_full, data_valid_full, data_discov_reduced, data_valid_reduced):
    bundle.meta = bundle.meta.assign(**{"included": lambda x: x["name"].map(included_in)})

In [14]:
# Save full and reduced data and meta-information about variables
data_full.save(output_folder / "data_full")

# Save full and reduced discovery and validation data
data_discov_full.save(output_folder / "data_discov_full")
data_valid_full.save(output_folder / "data_valid_full")
data_discov_reduced.save(output_folder / "data_discov_reduced")
data_valid_reduced.save(output_folder / "data_valid_reduced")

Saved dataset at data/preprocessed/data_full.
Saved dataset at data/preprocessed/data_discov_full.
Saved dataset at data/preprocessed/data_valid_full.
Saved dataset at data/preprocessed/data_discov_reduced.
Saved dataset at data/preprocessed/data_valid_reduced.


In [30]:
print(data_full.data["hne"].cat.categories)

Index(['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13'], dtype='str', name='hne')


In [31]:
print(data_discov_reduced.data.shape, data_valid_reduced.data.shape) 

(8835, 48) (8697, 49)


In [32]:
print(data_full.data.shape)   

(17532, 121)


In [33]:
print(data_full.na.loc[discov_mask, ["fag_sch", "fr12_meth1"]].apply(pd.Series.value_counts))
print(data_full.na.loc[~discov_mask, ["dsm_ta"]].value_counts())

              fag_sch  fr12_meth1
rule imputed     7011        8828
not included     1824           7
observed            0           0
allowed skip        0           0
no answer           0           0
imputed skip        0           0
dsm_ta      
rule imputed    6982
not included    1715
observed           0
allowed skip       0
no answer          0
imputed skip       0
Name: count, dtype: int64
